# 02 Pipeline

A compact, cloneable ETL: prep decides, physical IO executes, and checks plus metadata registration stay visible.

## Tested with FabricOps

The previous baseline was run in Microsoft Fabric with FabricOps v0.2.0 by Voyce on 6 Aug 2026. This redesigned workflow has local structural and public-API compatibility validation only; run it in your configured Fabric workspace before treating it as runtime-validated.

# 0. Environment

Load the shared Fabric configuration, public APIs, and only the two registries needed across Extract blocks.

In [ ]:
%run 00_env_config

In [ ]:
from pyspark.sql import functions as F

from fabricops_kit import (
    check_dq,
    check_freshness,
    check_schema,
    profile_and_register_table,
    read_lakehouse_table,
    read_pipeline_prep,
    read_warehouse_query,
    write_lakehouse_table,
    write_pipeline_prep,
    widget_select_data_contract,
    widget_view_catalogue,
)

EXTRACT_PREPS = {}
EXTRACT_DFS = {}
PIPELINE_SHOULD_RUN = True

## Notebook controls

Use one optional Data Contract selector and one Catalogue view. FabricOps uses the notebook name as the stable logical Lineage identity across environments; runtime notebook, workspace, and environment IDs remain audit context.

In [ ]:
# Step 2: leave False for the first run, before Lineage and frozen contracts exist.
# Step 4: set True to select one frozen contract for every lineage-linked table_id.
VALIDATE_DATA_CONTRACTS = False
CONTRACT_SELECTION = (
    widget_select_data_contract(spark_session=spark)
    if VALIDATE_DATA_CONTRACTS
    else None
)

catalogue_widget = widget_view_catalogue(mode="explore", spark_session=spark)

## How to read the blocks

Each Extract or Load block defines physical identity and strategy—not `table_id`. Prep deterministically resolves `table_id` and processing scope. The visible reader or writer then performs physical IO.

In [ ]:
# Orders uses this Load identity to resolve authoritative successful progress.
ORDERS_PROGRESS_LOAD = {
    "target_target": "unified",
    "target_schema": "demo",
    "target_table": "orders",
}

# E. Extract

Each Extract follows **define → prep → pre-read check → physical read → data checks → profile/register → store → view**.

## EXTRACT 1 — Orders

Incremental Lakehouse rows bounded by the successful Load watermark.

In [ ]:
EXTRACT = 1
EXTRACT_NAME = "Orders"
EXTRACT_TARGET = "source"
EXTRACT_SCHEMA = "demo"
EXTRACT_TABLE = "orders"
EXTRACT_READ_STRATEGY = "incremental_watermark"
EXTRACT_WATERMARK_COLUMN = "modified_datetime"

In [ ]:
extract_prep = read_pipeline_prep(
    source_target=EXTRACT_TARGET,
    source_schema=EXTRACT_SCHEMA,
    source_table=EXTRACT_TABLE,
    source_read_strategy=EXTRACT_READ_STRATEGY,
    source_watermark_column=EXTRACT_WATERMARK_COLUMN,
    **ORDERS_PROGRESS_LOAD,
)
EXTRACT_TABLE_ID = extract_prep["table_id"]

if extract_prep["observation"] is not None:
    check_freshness(
        extract_prep["observation"],
        table_id=EXTRACT_TABLE_ID,
        enabled=VALIDATE_DATA_CONTRACTS,
        raise_on_failure=True,
    )

PIPELINE_SHOULD_RUN = extract_prep["read_mode"] != "skip"
if PIPELINE_SHOULD_RUN:
    extract_df = read_lakehouse_table(
        table_id=EXTRACT_TABLE_ID,
        spark_session=spark,
        processing_scope=extract_prep["scope"],
    )
    check_schema(
        EXTRACT_TABLE_ID,
        dataframe=extract_df,
        enabled=VALIDATE_DATA_CONTRACTS,
        raise_on_failure=True,
    )
    check_dq(
        extract_df,
        table_id=EXTRACT_TABLE_ID,
        enabled=VALIDATE_DATA_CONTRACTS,
        raise_on_failure=True,
    )
    extract_profile = profile_and_register_table(
        extract_df,
        profile_role="source",
        table=extract_prep["source"],
        processing_scope=extract_prep["scope"],
    )
    display(extract_profile)

    EXTRACT_PREPS[EXTRACT] = extract_prep
    EXTRACT_DFS[EXTRACT] = extract_df
    catalogue_widget["show"](table_id=EXTRACT_TABLE_ID)
else:
    print("Orders is unchanged; physical reads, transforms, and writes are skipped.")

## EXTRACT 2 — Products

A complete Lakehouse reference read with the same visible stages.

In [ ]:
EXTRACT = 2
EXTRACT_NAME = "Products"
EXTRACT_TARGET = "source"
EXTRACT_SCHEMA = "demo"
EXTRACT_TABLE = "products"
EXTRACT_READ_STRATEGY = "full_dataset"

In [ ]:
if PIPELINE_SHOULD_RUN:
    extract_prep = read_pipeline_prep(
        source_target=EXTRACT_TARGET,
        source_schema=EXTRACT_SCHEMA,
        source_table=EXTRACT_TABLE,
        source_read_strategy=EXTRACT_READ_STRATEGY,
    )
    EXTRACT_TABLE_ID = extract_prep["table_id"]

    extract_df = read_lakehouse_table(
        table_id=EXTRACT_TABLE_ID,
        spark_session=spark,
        processing_scope=extract_prep["scope"],
    )
    check_schema(
        EXTRACT_TABLE_ID,
        dataframe=extract_df,
        enabled=VALIDATE_DATA_CONTRACTS,
        raise_on_failure=True,
    )
    check_dq(
        extract_df,
        table_id=EXTRACT_TABLE_ID,
        enabled=VALIDATE_DATA_CONTRACTS,
        raise_on_failure=True,
    )
    extract_profile = profile_and_register_table(
        extract_df,
        profile_role="source",
        table=extract_prep["source"],
        processing_scope=extract_prep["scope"],
    )
    display(extract_profile)

    EXTRACT_PREPS[EXTRACT] = extract_prep
    EXTRACT_DFS[EXTRACT] = extract_df
    catalogue_widget["show"](table_id=EXTRACT_TABLE_ID)

## EXTRACT 3 — Order History

The SQL stays visible and executes in the Warehouse; its aggregate receives a diagnostic profile.

In [ ]:
EXTRACT = 3
EXTRACT_NAME = "Order History"
EXTRACT_TARGET = "product"
EXTRACT_SCHEMA = "demo"
EXTRACT_TABLE = "order_history"
EXTRACT_READ_STRATEGY = "full_dataset"

EXTRACT_QUERY = """
SELECT
    customer_id,
    COUNT(*) AS historical_order_count,
    SUM(net_amount) AS historical_net_amount,
    MAX(order_datetime) AS latest_historical_order_datetime
FROM demo.order_history
GROUP BY customer_id
"""

In [ ]:
if PIPELINE_SHOULD_RUN:
    extract_prep = read_pipeline_prep(
        source_target=EXTRACT_TARGET,
        source_schema=EXTRACT_SCHEMA,
        source_table=EXTRACT_TABLE,
        source_read_strategy=EXTRACT_READ_STRATEGY,
    )
    EXTRACT_TABLE_ID = extract_prep["table_id"]

    extract_df = read_warehouse_query(
        EXTRACT_QUERY,
        target=extract_prep["source"]["target"],
        spark_session=spark,
    )
    check_schema(
        EXTRACT_TABLE_ID,
        dataframe=extract_df,
        enabled=VALIDATE_DATA_CONTRACTS,
        raise_on_failure=True,
    )
    check_dq(
        extract_df,
        table_id=EXTRACT_TABLE_ID,
        enabled=VALIDATE_DATA_CONTRACTS,
        raise_on_failure=True,
    )
    extract_profile = profile_and_register_table(
        extract_df,
        profile_role="source",
        table=extract_prep["source"],
        processing_scope=extract_prep["scope"],
        complete_table=False,
    )
    display(extract_profile)

    EXTRACT_PREPS[EXTRACT] = extract_prep
    EXTRACT_DFS[EXTRACT] = extract_df
    catalogue_widget["show"](table_id=EXTRACT_TABLE_ID)

### Profile semantics

Complete physical reads update canonical `METADATA_DATA_CATALOGUE`, `METADATA_DATA_PROFILED`, and eligible `METADATA_DATA_PROFILED_FREQUENCY` snapshots. Incremental subsets and custom query results return diagnostic profiles without replacing canonical full-table metadata. Extract and Load participation meet in `METADATA_DATA_LINEAGE` during write preparation.

# T. Transform

**Business transformation is project-owned PySpark.** FabricOps does not hide these joins or calculations.

In [ ]:
if PIPELINE_SHOULD_RUN:
    orders_df = EXTRACT_DFS[1].alias("orders")
    products_df = EXTRACT_DFS[2].alias("products")
    history_df = EXTRACT_DFS[3].alias("history")

    transformed_df = (
        orders_df
        .join(products_df, on="product_id", how="left")
        .join(history_df, on="customer_id", how="left")
        .withColumn(
            "order_net_amount",
            F.round(F.col("quantity") * F.col("unit_price") * (F.lit(1.0) - F.col("discount")), 2),
        )
        .fillna({"historical_order_count": 0, "historical_net_amount": 0.0})
        .select(
            "order_id", "customer_id", "order_datetime", "modified_datetime",
            "product_id", "product_name", "product_category", "quantity", "unit_price",
            "discount", "order_net_amount", "order_status", "shipping_country",
            "historical_order_count", "historical_net_amount", "latest_historical_order_datetime",
        )
    )
    display(transformed_df)

# L. Load

## LOAD 1 — Curated Orders

The Load follows **define → prep → checks → physical write → published read → profile/register → view**.

In [ ]:
if PIPELINE_SHOULD_RUN:
    LOAD = 1
    LOAD_NAME = "Curated Orders"
    LOAD_TARGET = "unified"
    LOAD_SCHEMA = "demo"
    LOAD_TABLE = "orders"
    LOAD_STRATEGY = "scd1"
    LOAD_PARAMETERS = {"key_columns": ["order_id"]}
    load_df = transformed_df

In [ ]:
if PIPELINE_SHOULD_RUN:
    load_prep = write_pipeline_prep(
        load_df,
        target=LOAD_TARGET,
        schema=LOAD_SCHEMA,
        table_name=LOAD_TABLE,
        load_strategy=LOAD_STRATEGY,
        load_strategy_parameters=LOAD_PARAMETERS,
        source_preps=[
            EXTRACT_PREPS[1],
            EXTRACT_PREPS[2],
            EXTRACT_PREPS[3],
        ],
    )
    LOAD_TABLE_ID = load_prep["target"]["table_id"]
    prepared_df = load_prep["df"].persist()

In [ ]:
if PIPELINE_SHOULD_RUN:
    check_schema(
        LOAD_TABLE_ID,
        dataframe=load_df,
        enabled=VALIDATE_DATA_CONTRACTS,
        raise_on_failure=True,
    )
    check_dq(
        load_df,
        table_id=LOAD_TABLE_ID,
        enabled=VALIDATE_DATA_CONTRACTS,
        raise_on_failure=True,
    )

### Physical write

`repartition_by=4` uses distributed Spark execution; it does not create Python threads or independent writers.

In [ ]:
if PIPELINE_SHOULD_RUN:
    write_lakehouse_table(
        prepared_df,
        load_prep["target"]["table_name"],
        target=load_prep["target"]["target"],
        schema=load_prep["target"]["schema"],
        mode=load_prep["mode"],
        options=load_prep["options"],
        load_strategy=load_prep["load_strategy"],
        load_strategy_parameters=load_prep["load_strategy_parameters"],
        processing_scope=load_prep["scope"],
        repartition_by=4,
    )
    prepared_df.unpersist()

### Published state and stored metadata

Read the published table back before recording its canonical full-table profile.

In [ ]:
if PIPELINE_SHOULD_RUN:
    published_df = read_lakehouse_table(
        table_id=LOAD_TABLE_ID,
        spark_session=spark,
    )
    load_profile = profile_and_register_table(
        published_df,
        profile_role="target",
        table=load_prep["target"],
        load_strategy=load_prep["load_strategy"],
        load_strategy_parameters=load_prep["load_strategy_parameters"],
    )
    display(load_profile)
    catalogue_widget["show"](table_id=LOAD_TABLE_ID)